In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads HF_TOKEN from .env file

In [3]:
! pip install -q youtube-transcript-api langchain-community langchain-huggingface huggingface_hub faiss-cpu sentence-transformers python-dotenv


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpoint , HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\somil\AppData\Local\Temp\ipykernel_1020\822431559.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# 1. Document Ingestion

In [90]:
video_id = "7ARBJQn6QkM"

ytt_api = YouTubeTranscriptApi()

try:
    transcript = ytt_api.fetch(video_id, languages=["en"])

    text = " ".join(snippet.text for snippet in transcript)
    print(text)

except TranscriptsDisabled:
    print("No captions available for this video.")

At some point, you have to believe something. 
We've reinvented computing as we know it. What is the vision for what you see coming next? We 
asked ourselves, if it can do this, how far can it go? How do we get from the robots that 
we have now to the future world that you see? Cleo, everything that moves will be 
robotic someday and it will be soon. We invested tens of billions of dollars before 
it really happened. No that's very good, you did some research! But the big breakthrough 
I would say is when we... That's Jensen Huang, and whether you know it or not
his decisions are shaping your future. He's the CEO of NVIDIA, the company that skyrocketed over the past few
years to become one of the most valuable companies in the world because they led a fundamental shift 
in how computers work unleashing this current explosion of what's possible with technology. 
"NVIDIA's done it again!" We found ourselves being one of the most important technology companies in 
the world and potentiall

In [91]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="At some point, you have to believe something.\xa0\nWe've reinvented computing as we know it. What", start=0.08, duration=3.68), FetchedTranscriptSnippet(text='is the vision for what you see coming next? We\xa0\nasked ourselves, if it can do this, how far can', start=3.76, duration=4.72), FetchedTranscriptSnippet(text='it go? How do we get from the robots that\xa0\nwe have now to the future world that you', start=8.48, duration=4.64), FetchedTranscriptSnippet(text='see? Cleo, everything that moves will be\xa0\nrobotic someday and it will be soon. We', start=13.12, duration=4.08), FetchedTranscriptSnippet(text="invested tens of billions of dollars before\xa0\nit really happened. No that's very good, you", start=17.2, duration=5.04), FetchedTranscriptSnippet(text='did some research! But the big breakthrough\xa0\nI would say is when we...', start=22.24, duration=5.994), FetchedTranscriptSnippet(text="That's Jensen Huang, and whethe

# 2. Text Splitting

In [92]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([text])

In [93]:
len(chunks)

71

In [94]:
chunks[50]

Document(metadata={}, page_content="the limits and so we discover the limits together. Now we do the same thing in system engineering and\xa0\ncooling systems. It turns out plumbing is really important to us because of liquid cooling.\xa0\nAnd maybe fans are really important to us because of air cooling and we're trying to design\xa0\nthese fans in a way almost like you know they're aerodynamically sound so that we could pass the\xa0\nhighest volume of air, make the least amount of noise. So we have aerodynamics engineers in our\ncompany. And so even though even though we don't make 'em, we design them and we have to deep\xa0\nexpertise of knowing how to have them made. And and from that we try to push the\xa0\nlimits. One of the themes of this conversation is that you are a person who makes big bets on the\xa0\nfuture and time and time again you've been right about those bets. We've talked about GPUs, we've\xa0\ntalked about CUDA, we've talked about bets you've made in AI - self-drivi

# 3. Embedding generation and store in vector


In [96]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3796.46it/s]


In [97]:
vector_store = FAISS.from_documents(chunks, embedding)

In [98]:
vector_store.index_to_docstore_id

{0: '7a03e24d-a239-4323-8a9f-e91d73f0a756',
 1: '778d4214-fe07-4b7e-b723-dcd8883cf032',
 2: 'c786f34f-1f6e-445e-96d4-9cb506cb84e5',
 3: 'b8c09dba-2b1c-400d-b89b-98309eddac8e',
 4: '7b0de7fc-753f-4e96-a145-fcd103992a09',
 5: '4139db43-1e5d-4bc9-bea7-4e7050279f1d',
 6: '3bb4ae29-2852-4021-9db0-681d3e666f33',
 7: 'ae3de34c-cf92-430a-875f-f8707bddcc1a',
 8: 'be1cb746-1093-4d34-8a45-41c6c3fe50a8',
 9: 'b941b1c6-2641-4134-88e4-5e66355f673c',
 10: '4d97046b-685f-4502-b237-cf9dd9b2cb31',
 11: 'fef2fb84-d9d4-407d-aa41-9cb251f36232',
 12: '065ac7a0-8d50-4d1a-a785-f446e5caf978',
 13: '8e5cc9b1-bb68-4b2b-a2ea-31e79a238526',
 14: 'cfd106a3-44f3-4c4a-b97a-fff679307121',
 15: 'd0d137e0-46f0-4b22-910c-f15d8bf5d479',
 16: 'd35b4e38-5bf4-4d39-addd-d721243fced7',
 17: 'f70b1f9c-2441-467d-9a89-ec575f2380f0',
 18: '30f4d049-b5ab-45f3-8e22-2f0bb52c8988',
 19: 'd64b2d0d-5942-49d3-9953-0b0da1d507b1',
 20: 'e769ad91-b7d8-4a65-9191-29603308b816',
 21: 'b9f4e148-8a5f-4787-bae9-bc31686b2254',
 22: '3322c267-2ed3-

In [99]:
vector_store.get_by_ids(['ed0ca507-ad78-41b1-b23a-ce6b00514d88'])

[Document(id='ed0ca507-ad78-41b1-b23a-ce6b00514d88', metadata={}, page_content="And I'm very curious for you, you've you've been doing this a long time, it feels like there's\xa0\nso much that you've described in this vision ahead, what would the theme be that you would\xa0\nwant people to say about what you're trying to do? Very simply, they made an extraordinary impact.\xa0\nI think that we're fortunate because of some core beliefs a long time ago and sticking with\xa0\nthose core beliefs and building upon them we found ourselves today being one of\xa0\nthe most, one of the many most important and consequential technology companies in\nthe world and potentially ever. And so we take that responsibility very\xa0seriously.\nWe work hard to make sure that the capabilities that we've created are\xa0\navailable to large companies as well as individual researchers and developers, across\xa0\nevery field of science no matter profitable or not, big or small, famous or otherwise. \nAnd it's be

In [100]:
print(vector_store.docstore._dict)

{'7a03e24d-a239-4323-8a9f-e91d73f0a756': Document(id='7a03e24d-a239-4323-8a9f-e91d73f0a756', metadata={}, page_content='At some point, you have to believe something.\xa0\nWe\'ve reinvented computing as we know it. What is the vision for what you see coming next? We\xa0\nasked ourselves, if it can do this, how far can it go? How do we get from the robots that\xa0\nwe have now to the future world that you see? Cleo, everything that moves will be\xa0\nrobotic someday and it will be soon. We invested tens of billions of dollars before\xa0\nit really happened. No that\'s very good, you did some research! But the big breakthrough\xa0\nI would say is when we... That\'s Jensen Huang, and whether you know it or not\nhis decisions are\xa0shaping your future. He\'s the CEO of NVIDIA, the company that skyrocketed over the past few\nyears\xa0to become one of the most valuable companies in the world because they led a fundamental shift\xa0\nin how computers work unleashing this current explosion of 

# 4. Retriever

In [103]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [104]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D800D5CCD0>, search_kwargs={'k': 3})

In [105]:
retriever.invoke('What is GPU?')

[Document(id='ae3de34c-cf92-430a-875f-f8707bddcc1a', metadata={}, page_content='time or parallel processing on a GPU. "3... 2... 1..." So Nvidia unlocks all of this new power\nfor\xa0video games. Why gaming first? The video games requires parallel processing for processing\xa0\n3D graphics and we chose video games because, one, we loved the application, it\'s a simulation\xa0\nof virtual worlds and who doesn\'t want to go to virtual worlds and we had the good observation\xa0\nthat video games has potential to be the largest market for for entertainment ever. And it turned\xa0\nout to be true. And having it being a large market is important because the technology is complicated\xa0\nand if we had a large market, our R&D budget could be large, we could create new technology. And that\xa0\nflywheel between technology and market and greater technology was really the flywheel that\xa0\ngot NVIDIA to become one of the most important technology companies in the world. It was all\xa0\nbecause 

# 5 . Augmentation


In [106]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
)

chat = ChatHuggingFace(llm=llm)


In [107]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [108]:
question          = "What is GPU?"
retrieved_docs    = retriever.invoke(question)

In [109]:
retrieved_docs

[Document(id='ae3de34c-cf92-430a-875f-f8707bddcc1a', metadata={}, page_content='time or parallel processing on a GPU. "3... 2... 1..." So Nvidia unlocks all of this new power\nfor\xa0video games. Why gaming first? The video games requires parallel processing for processing\xa0\n3D graphics and we chose video games because, one, we loved the application, it\'s a simulation\xa0\nof virtual worlds and who doesn\'t want to go to virtual worlds and we had the good observation\xa0\nthat video games has potential to be the largest market for for entertainment ever. And it turned\xa0\nout to be true. And having it being a large market is important because the technology is complicated\xa0\nand if we had a large market, our R&D budget could be large, we could create new technology. And that\xa0\nflywheel between technology and market and greater technology was really the flywheel that\xa0\ngot NVIDIA to become one of the most important technology companies in the world. It was all\xa0\nbecause 

In [110]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'time or parallel processing on a GPU. "3... 2... 1..." So Nvidia unlocks all of this new power\nfor\xa0video games. Why gaming first? The video games requires parallel processing for processing\xa0\n3D graphics and we chose video games because, one, we loved the application, it\'s a simulation\xa0\nof virtual worlds and who doesn\'t want to go to virtual worlds and we had the good observation\xa0\nthat video games has potential to be the largest market for for entertainment ever. And it turned\xa0\nout to be true. And having it being a large market is important because the technology is complicated\xa0\nand if we had a large market, our R&D budget could be large, we could create new technology. And that\xa0\nflywheel between technology and market and greater technology was really the flywheel that\xa0\ngot NVIDIA to become one of the most important technology companies in the world. It was all\xa0\nbecause of video games. I\'ve heard you say that GPUs were a time machine? Yeah. Could 

In [111]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

# 6. Generation

In [112]:
response = chat.invoke(final_prompt)
print(response.content)

A GPU, or Graphics Processing Unit, is a type of processor designed to manipulate and alter memory to accelerate the creation of images in a frame buffer intended for output to a display device. GPUs are very efficient at handling parallel processing tasks, which are common in 3D graphics and video games. This capability was recognized by Nvidia, who developed GPUs that could also be used for general-purpose computing, beyond just graphics processing.


# Building chain

In [113]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [114]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [115]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [116]:
parallel_chain.invoke('What is GPU?')

{'context': 'time or parallel processing on a GPU. "3... 2... 1..." So Nvidia unlocks all of this new power\nfor\xa0video games. Why gaming first? The video games requires parallel processing for processing\xa0\n3D graphics and we chose video games because, one, we loved the application, it\'s a simulation\xa0\nof virtual worlds and who doesn\'t want to go to virtual worlds and we had the good observation\xa0\nthat video games has potential to be the largest market for for entertainment ever. And it turned\xa0\nout to be true. And having it being a large market is important because the technology is complicated\xa0\nand if we had a large market, our R&D budget could be large, we could create new technology. And that\xa0\nflywheel between technology and market and greater technology was really the flywheel that\xa0\ngot NVIDIA to become one of the most important technology companies in the world. It was all\xa0\nbecause of video games. I\'ve heard you say that GPUs were a time machine? 

In [117]:
parser = StrOutputParser()

In [118]:
chain = parallel_chain | prompt | chat | parser

In [119]:
chain.invoke('what is GPU?')

'A GPU, or Graphics Processing Unit, is a specialized electronic circuit designed to rapidly manipulate and alter memory to accelerate the creation of images in a frame buffer intended for output to a display device. In the context of the provided transcript, GPUs were particularly noted for their ability to perform parallel processing, which is crucial for handling the complex calculations required for 3D graphics in video games. This capability also led to GPUs being used in a wide range of other applications beyond gaming.'

In [120]:
from sklearn.metrics.pairwise import cosine_similarity


# Two texts
text1 = "A GPU, or Graphics Processing Unit, is a type of processor designed to manipulate and alter memory to accelerate the creation of images in a frame buffer intended for output to a display device. GPUs are very efficient at handling parallel processing tasks, which are common in 3D graphics and video games. This capability was recognized by Nvidia, who developed GPUs that could also be used for general-purpose computing, beyond just graphics processing."
text2 = "'A GPU, or Graphics Processing Unit, is a specialized electronic circuit designed to rapidly manipulate and alter memory to accelerate the creation of images in a frame buffer intended for output to a display device. In the context of the provided transcript, GPUs were particularly noted for their ability to perform parallel processing, which is crucial for handling the complex calculations required for 3D graphics in video games. This capability also led to GPUs being used in a wide range of other applications beyond gaming.'"

# Generate embeddings
embedding1 = embedding.embed_query(text1)
embedding2 = embedding.embed_query(text2)

# Compute cosine similarity
similarity = cosine_similarity(
    [embedding1],
    [embedding2]
)[0][0]

print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.9657
